# Bài thực hành 01 – Thu thập dữ liệu ô tô trên Bonbanh.com

**Họ tên:** Nguyễn Phi Nhung  **MSSV:** 245212

## 1. Giới thiệu bộ dữ liệu

- **Nguồn:** [bonbanh.com](https://bonbanh.com), trang rao bán ô tô cũ và mới lớn tại Việt Nam.
- **Đơn vị quan sát:** mỗi dòng là một tin đăng bán xe.
- **Mục đích:** có một bảng dữ liệu gồm đặc điểm xe (hãng, năm sản xuất, số km, hộp số, nhiên liệu...) và giá bán, để dùng cho các bài sau của môn Học máy thống kê (ví dụ dự đoán giá xe, phân loại, phân cụm).

## 2. Cách thu thập dữ liệu

1. **Trang được cào:** các trang danh sách theo hãng, dạng `bonbanh.com/oto/<hãng>/page,N`. Mỗi tin trong trang danh sách đã có đủ thông tin cần dùng nên không phải vào từng trang chi tiết.
2. **Công cụ:** thư viện `requests` tải HTML, `BeautifulSoup` đọc HTML, biểu thức chính quy (`re`) tách các trường từ chuỗi, `pandas` lưu file. (Nếu `requests` không nhận được danh sách xe thì dùng thêm `playwright` để mở trình duyệt thật.)
3. **Phạm vi:** 38 hãng xe, mỗi hãng lấy tối đa 30 trang đầu (20 tin mỗi trang - tối đa khoảng 600 tin mỗi hãng). Hãng ít tin thì hết trang sớm hơn.
4. **Lịch sự với trang web:** nghỉ khoảng 1 giây giữa các lần tải trang. Không lưu tên và số điện thoại người bán.
5. **Kết quả lưu ở:** `cars.csv` (toàn bộ)

**Hạn chế cần lưu ý:** dữ liệu là 30 trang đầu của mỗi hãng theo thứ tự mặc định của trang web, nên không phải mẫu ngẫu nhiên. Hãng lớn (Toyota, Hyundai...) có nhiều tin hơn thực tế vẫn chỉ lấy tối đa như hãng nhỏ hơn có nhiều tin. Giá là giá người bán rao, không phải giá giao dịch.

## 3. Mô tả các cột

| Cột | Ý nghĩa |
|---|---|
| `id` | Mã tin đăng | 
| `condition` | Tình trạng: cũ / mới |
| `brand`, `model`, `variant` | Hãng, dòng xe, phiên bản |
| `year` | Năm sản xuất |
| `price_million` | Giá rao bán (triệu đồng) |
| `province` | Tỉnh / thành phố của tin đăng |
| `origin` | Xuất xứ: nhập khẩu / trong nước |
| `color` | Màu xe |
| `fuel` | Nhiên liệu: xăng / dầu / điện / hybrid |
| `engine_l` | Dung tích động cơ (lít) |
| `gearbox` | Hộp số: tự động / tay (sàn) |
| `seats` | Số chỗ ngồi |
| `km` | Số km đã đi (xe mới thường không có) |
| `url` | Đường dẫn tin đăng |

## 4. Thống kê cơ bản

In [10]:
import pandas as pd
import matplotlib.pyplot as plt

In [11]:
df = pd.read_csv("cars_raw.csv")
print("Số dòng, số cột:", df.shape)
df.head()

Số dòng, số cột: (10746, 16)


,id,condition,brand,model,variant,year,price_million,province,origin,color,fuel,engine_l,gearbox,seats,km,url
0,7023149,cũ,toyota,vios,1.5g,2017,310.0,Hà Nội,trong nước,cát,xăng,1.5,tự động,5,171000.0,https://bonbanh.com/xe-toyota-vios-1.5g-2017-7...
1,6418608,cũ,toyota,camry,2.5q,2017,576.0,Hà Nội,trong nước,đen,xăng,2.5,tự động,5,NaN,https://bonbanh.com/xe-toyota-camry-2.5q-2017-...
2,7024014,cũ,toyota,camry,2.5q,2015,505.0,Hà Nội,trong nước,đen,xăng,2.5,tự động,5,100000.0,https://bonbanh.com/xe-toyota-camry-2.5q-2015-...
3,6938711,cũ,toyota,innova,2.0g,2015,269.0,Hà Nội,trong nước,bạc,xăng,2.0,tự động,8,120000.0,https://bonbanh.com/xe-toyota-innova-2.0g-2015...
4,6991912,mới,toyota,veloz,cross top 1.5 cvt,2026,540.0,Bình Dương,trong nước,trắng,xăng,1.5,tự động,7,NaN,https://bonbanh.com/xe-toyota-veloz-cross-top-...


In [12]:
df.dtypes

id                 int64
condition            str
brand                str
model                str
variant              str
year               int64
price_million    float64
province             str
origin               str
color                str
fuel                 str
engine_l         float64
gearbox              str
seats              int64
km               float64
url                  str
dtype: object

In [13]:
missing = pd.DataFrame({
    "so_o_thieu": df.isna().sum(),
    "ti_le_%": (df.isna().mean() * 100).round(2),
})
print(missing[missing["so_o_thieu"] > 0].sort_values("ti_le_%", ascending=False))

          so_o_thieu  ti_le_%
km              3324    30.93
engine_l         681     6.34
color             17     0.16
variant           13     0.12
fuel               1     0.01
gearbox            1     0.01
